In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI

In [2]:
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API key was found")
elif not api_key.startswith('sk-proj'):
    print("API key is found but not the right API key")
else:
    print("Right API Key is found")

Right API Key is found


In [3]:
message = "Hello GPT!, this is my first message"

messages = [
    {"role":"user", "content":message}
]
messages

[{'role': 'user', 'content': 'Hello GPT!, this is my first message'}]

In [5]:
openai = OpenAI()

response = openai.chat.completions.create(
    model = "gpt-4",
    messages=messages
)

response.choices[0].message.content

"Hello! I'm glad to meet you. I'm an AI developed by OpenAI, how can I assist you today?"

### System Prompt

In [6]:
system_prompt = """
You are an assistant that analyzes the contents of a website,
and provides a short summary , ignoring text that might redirect to other pages.
Respond in markdown"""

In [18]:
user_prompt = """Here is a website.
Provide a short summary of this website in markdown.
If it includes news or announcements then summarize them as well."""

In [9]:
messages = [
    {"role":"system", "content":"You are a snarky assistant"},
    {"role":"user", "content":"What is 10+10?"}
]

response = openai.chat.completions.create(
    model = "gpt-4",
    messages=messages
)


response.choices[0].message.content

"Wow, this one's really tough! Let's see, by my calculations... it's 20. You're really putting my advanced AI technology to good use."

In [19]:
def message_for(website):
    return [
        {"role":"system", "content":system_prompt},
        {"role":"user", "content":user_prompt + website}
    ]

In [22]:
import requests
from bs4 import BeautifulSoup

def fetch_website_contents(url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()  # Raise error for bad status
        
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Extract visible text
        return soup.get_text(separator="\n", strip=True)
    
    except requests.exceptions.RequestException as e:
        return f"Error fetching website: {e}"

In [20]:
message_for("https://www.tcs.com")

[{'role': 'system',
  'content': '\nYou are an assistant that analyzes the contents of a website,\nand provides a short summary , ignoring text that might redirect to other pages.\nRespond in markdown'},
 {'role': 'user',
  'content': 'Here is a website.\nProvide a short summary of this website in markdown.\nIf it includes news or announcements then summarize them as well.https://www.tcs.com'}]

In [25]:
def summarize(url):

    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model="gpt-4.1",
        messages = message_for(website)
    )

    return response.choices[0].message.content

In [27]:
from IPython.display import display, Markdown
display(Markdown(summarize("https://www.goal.com/en-gb")))

```markdown
# Goal.com UK – Website Summary

**Goal.com UK** is a comprehensive football (soccer) news platform providing up-to-date coverage on matches, scores, transfers, analysis, and features across global football. The website serves fans of both men's and women's football, focusing on major clubs, leagues, international teams, and emerging young talents.

## Core Features
- **Live Scores & Results:** Real-time updates on ongoing matches, including international friendlies and club competitions.
- **Latest News:** Breaking news, match reports, and updates on transfer activities.
- **Transfers:** Insight and analysis on the latest player transfers and rumors.
- **Opinion & Analysis:** Expert commentary on tactics, players, clubs, and football culture.
- **Player Ratings & Rankings:** Performance evaluations and lists, including features on upcoming “wonderkids.”
- **Women's Football:** Dedicated section covering major women's leagues, tournaments, and players.
- **Betting Guides:** Tips, odds explanations, and guides for betting on football matches.

## News & Announcements (Highlights)
- **Transfer Updates:**
  - Casemiro has confirmed his departure from Manchester United, with interest from Inter.
  - Dani Ceballos is rumored to be leaving Real Madrid for another La Liga club.
  - Lautaro Di Lollo, a young Boca Juniors defender, is attracting European interest.
- **Premier League News:**
  - Tottenham Hotspur is considering Sean Dyche as a new manager to aid a difficult season.
  - Alexander Isak has joined Liverpool in a record deal and is expected to prove his value despite a challenging start.
- **Women's Football:**
  - Arsenal Women's captain Kim Little has extended her contract with the club.
  - Manchester United’s Ella Toone is sidelined due to injury, affecting both club and England fixtures.
  - Chelsea Women’s manager Sonia Bompastor criticized refereeing standards after a key UWCL defeat.
- **Youth & Wonderkids:**
  - The annual NXGN lists have been updated for 2026, highlighting the world’s top male and female teenage talents, including repeated recognition for Lamine Yamal.
- **Other Headlines:**
  - Lionel Messi’s international future draws speculation.
  - Manchester United Women are facing a crucial week likely to define their season.
  - New Champions League final kickoff time at 6pm for broader fan and host city appeal.

## Extra Content
- **Videos and Podcasts:** Interviews with players and experts, football quizzes, and special features.
- **Entertainment & Lifestyle:** Football culture, kit and boot reviews, and stories of football fandom and personalities.
- **Betting Section:** Extensive coverage of betting predictions and offers.

**In summary, Goal.com UK is a leading destination for football fans seeking up-to-date news, live scores, expert analysis, and in-depth features on all aspects of the game, with dedicated coverage of both the men's and women's games and emerging stars.**
```